This document aims to accomplish the Task 2 "Balancedness: Matched vs Unmatched" questions. It uses the Step 1 store definitions as they are: a store is a WSLCB
**storefront** (address), the universe is the 542 storefronts with a license operating in some month of 2019-2021 by the dated WSLCB samples ([1e'], [6a]), a
storefront-month is **matched** when it has an approved Veraset label that month ([1e]), and a storefront is matched when any of its open months is ([6a]).

(a) Check the store history against a second source. The store history file from the task (`store_history_2018_2021.zip`, with
`store_month_activity_2018_2021.csv`) isn't in the project yet, so this uses the LCB's own archived "Sales Activity by License Number" files (monthly retail
sales by license, Jul 2014 - Aug 2020, Internet Archive copies in `raw-data/lcb-sales/`). The same code can take the store history file once it's here

(i) One row per storefront-month in 2019-01 - 2020-07 (the months with complete sales data): open by the WSLCB samples or not, and whether a license operating
there had positive sales. Label as ([9ai])

(ii) Licenses with retail sales in a month when no storefront has them operating (by [1e']), and where those licenses are. Label as ([9aii])

(b) Compare places. Among census places with at least one storefront, compare place characteristics across match-rate quartiles (terciles and quintiles as
robustness checks): population, population density, and urbanicity, median household income, unemployment rate, and poverty rate

(i) One row per place with its match rate and characteristics. Label as ([9bi])

(ii) Characteristics by match-rate group, for terciles, quartiles, and quintiles, with tests. Label as ([9bii])

(c) Compare stores. Matched vs. unmatched storefronts and storefront-months: opening cohort, months active and gap months, sales, address ambiguity (street,
neighborhood, and surrounding retail), and storefront size

(i) One row per storefront with every characteristic. Label as ([9ci])

(ii) Matched vs. unmatched storefronts, side by side, with tests, and a regression with all of them together. Label as ([9cii])

(iii) One row per storefront-month (open months, 2019-2021) and matched vs. unmatched storefront-months, including within-store comparisons. Label as ([9ciii])

(iv) A short list of storefronts to check by hand on Google Maps for storefront size. Label as ([9civ])

Notes:
- Census inputs (downloaded from www2.census.gov, no API key): ACS 2017-2021 5-year tables B19013 (median household income), B23025 (unemployment), B17001
  (poverty) by place; 2020 Census urban blocks (`2020_UA_BLOCKS.txt`); the 2020 block-to-place file (`BlockAssign_ST53_WA.zip`); TIGER/Line 2020 places and
  block groups for land area. Population is the 2020 Census count already in [3d] / `p12_blocks_2020.parquet`
- The ACS 2017-2021 window lines up with the 2019-2021 store window and uses the same 2020 place codes as [3d]
- Sales are self-reported net sales, the LCB's own caveat. Licence-level sales stop after Aug 2020 (and Aug 2020 is partial), so sales measures cover
  2019-01 - 2020-07 only
- Surrounding retail uses the SafeGraph POI file on disk (a current snapshot, not 2019-2021), so it describes today's surroundings
- Charts also get saved to "Task 2/Charts and Figures" as .html files

In [1]:
### IMPORTS AND PRELIMINARIES
import os, re, zipfile, warnings
import numpy as np
import pandas as pd
import polars as pl
import geopandas as gpd
import plotly.express as px
from scipy import stats
import statsmodels.formula.api as smf
warnings.filterwarnings("ignore")

STOREFRONT = ["street_address_norm", "city_norm"]
CENSUS = "raw-data/census"
DOWNLOADS = f"{CENSUS}/downloads"
SALES_FOLDER = "raw-data/lcb-sales"
POI_FOLDER = "raw-data/veraset-visits-wa/SafeGraph_POI"   # SafeGraph places file on disk (current snapshot)
CHART_FOLDER = "Task 2/Charts and Figures"
os.makedirs(CHART_FOLDER, exist_ok = True)

MONTHS = [f"{y}-{m:02d}" for y in (2019, 2020, 2021) for m in range(1, 13)]
SALES_MONTHS = [m for m in MONTHS if m <= "2020-07"]      # complete licence-level sales data

def show(fig, name):
    fig.show()
    fig.write_html(f"{CHART_FOLDER}/{name}.html", include_plotlyjs = "cdn")

def month_index(m):  # "2019-03" -> months since 2000-01
    return (int(m[:4]) - 2000) * 12 + int(m[5:7]) - 1

In [2]:
### LOADING: Step 1 store tables, used as they are
coverage = pl.read_parquet("processed-data/6a_storefront_coverage.parquet").to_pandas()                 # [6a] 542 storefronts
open_months = pl.read_parquet("processed-data/1e_storefront_months.parquet").filter(~pl.col("current_list_only")).to_pandas()   # [1e']
labels = pl.read_parquet("processed-data/1e_storefront_place_months.parquet")
covered = labels.filter(pl.col("approved")).select(STOREFRONT + ["month"]).unique().to_pandas().assign(covered = True)
locations = pl.read_parquet("processed-data/1a_wslcb_locations.parquet").to_pandas()                    # [1a] license x address
coords = pl.read_parquet("processed-data/1d_wslcb_location_coordinates.parquet").to_pandas()            # [1d]
places = pl.read_parquet("processed-data/3d_place_population_2020.parquet").to_pandas()                 # [3d]

# Storefront-months exactly as in [6a]: open months from [1e'], matched when an approved [1e] label exists that month
store_months = open_months.merge(covered, on = STOREFRONT + ["month"], how = "left")
store_months["covered"] = store_months.covered.fillna(False).astype(bool)
check = store_months.groupby(STOREFRONT).covered.sum().rename("c").reset_index().merge(coverage[STOREFRONT + ["covered_months"]], on = STOREFRONT)
print("storefront-months:", len(store_months), "| covered months agree with [6a]:", (check.c == check.covered_months).all())
print("storefronts:", len(coverage), "| matched:", coverage.matched.sum())

storefront-months: 17742 | covered months agree with [6a]: True
storefronts: 542 | matched: 427


In [3]:
### LOADING: LCB monthly retail sales by license (Jul 2014 - Aug 2020)
## Output: sales, one row per license and month with "sales" (net sales, dollars) and "excise"
def read_sales_recent(path):
    raw = pd.read_excel(path, sheet_name = "Retailers", header = None)
    header = raw.index[raw[0].astype(str).str.strip() == "License Number"][0]
    d = raw.iloc[header + 1:, :4].set_axis(["license", "period", "sales", "excise"], axis = 1)
    d = d[pd.to_numeric(d.license, errors = "coerce").notna()]
    d["month"] = pd.to_datetime(d.period.astype(str).str[:7], format = "%Y-%m").dt.strftime("%Y-%m")
    return d

def read_sales_old(path):  # one sheet, producers / processors / retailers blocks; retail rows follow the "RETAILERS" heading
    raw = pd.read_excel(path, header = None)
    start = raw.index[raw[0].astype(str).str.strip() == "RETAILERS"][0]
    d = raw.iloc[start:, :4].set_axis(["license", "period", "sales", "excise"], axis = 1)
    d = d[pd.to_numeric(d.license, errors = "coerce").notna()]
    d["month"] = pd.to_datetime(d.period.astype(str).str.split(" - ").str[0], format = "%m/%d/%Y").dt.strftime("%Y-%m")
    return d

sales = pd.concat([read_sales_old(f"{SALES_FOLDER}/tax_by_license_thru_2017-10-31.xlsx"),
                   read_sales_recent(f"{SALES_FOLDER}/sales_by_license_2020-10-02.xlsx")])
sales["license"] = sales.license.astype(int)
sales["sales"] = pd.to_numeric(sales.sales, errors = "coerce")
sales["excise"] = pd.to_numeric(sales.excise, errors = "coerce")
sales = sales.groupby(["license", "month"], as_index = False)[["sales", "excise"]].sum()   # a few license-months are reported in two rows
print("license-months:", len(sales), "| licenses:", sales.license.nunique(), "| months:", sales.month.min(), "-", sales.month.max())
print("license-months with zero or negative sales:", (sales.sales <= 0).sum())

license-months: 23669 | licenses: 563 | months: 2014-07 - 2020-08
license-months with zero or negative sales: 1


### PART A: the store history against the LCB's sales records

In [4]:
### PART A, SUBPART I
## Output: [9ai] one row per storefront and month in 2019-01 - 2020-07 that is open by [1e'] or has sales by a license that operates at the storefront in some month
##         Columns are STOREFRONT, "month", "open_wslcb" (a license is operating there that month by the dated WSLCB samples), "sales" (net sales of the
##         licenses operating there that month, or of the storefront's licenses for months it isn't open), "has_sales" (sales > 0), "covered" (approved
##         Veraset label that month), "matched" (the storefront is matched in [6a]), "status": "open, sales", "open, no sales record", "sales, not open by WSLCB"
## Name: 9ai_store_history_check.parquet
lic_store = open_months[STOREFRONT + ["month", "operating_licenses"]].explode("operating_licenses").rename(columns = {"operating_licenses": "license"})
lic_store["license"] = lic_store.license.astype(int)
both = lic_store.groupby(["license", "month"]).size()
print("license-months operating at 2+ storefronts at once (sales split equally):", (both > 1).sum())
lic_store = lic_store.merge(both.rename("n_storefronts").reset_index(), on = ["license", "month"])

s = sales[sales.month.isin(SALES_MONTHS)]
open_sales = (lic_store[lic_store.month.isin(SALES_MONTHS)]
    .merge(s, on = ["license", "month"], how = "left")
    .assign(sales = lambda d: d.sales / d.n_storefronts)
    .groupby(STOREFRONT + ["month"], as_index = False)
    .agg(sales = ("sales", lambda x: x.sum(min_count = 1)))
    .assign(open_wslcb = True))

# Months a storefront's own licenses (any license ever operating there) had sales, but no license is operating there by [1e']
store_licenses = lic_store[STOREFRONT + ["license"]].drop_duplicates()
lic_n_stores = store_licenses.groupby("license").size().rename("n_storefronts_ever")
extra = (store_licenses.merge(lic_n_stores, on = "license")
    .query("n_storefronts_ever == 1")     # only licenses that never move, so the sales are at this address
    .merge(s[s.sales > 0], on = "license")
    .merge(open_sales[STOREFRONT + ["month"]], on = STOREFRONT + ["month"], how = "left", indicator = True)
    .query("_merge == 'left_only'")
    .groupby(STOREFRONT + ["month"], as_index = False).agg(sales = ("sales", "sum"))
    .assign(open_wslcb = False))

history_check = (pd.concat([open_sales, extra])
    .merge(store_months[STOREFRONT + ["month", "covered"]], on = STOREFRONT + ["month"], how = "left")
    .merge(coverage[STOREFRONT + ["matched"]], on = STOREFRONT, how = "left"))
history_check["covered"] = history_check.covered.fillna(False).astype(bool)
history_check["has_sales"] = history_check.sales.fillna(0) > 0
history_check["status"] = np.select(
    [history_check.open_wslcb & history_check.has_sales, history_check.open_wslcb],
    ["open, sales", "open, no sales record"], "sales, not open by WSLCB")
history_check.to_parquet("processed-data/9ai_store_history_check.parquet")

t = history_check.groupby("status").agg(storefront_months = ("month", "size"), storefronts = ("street_address_norm", lambda x: x.nunique()))
t["share"] = t.storefront_months / t.storefront_months.sum()
display(t)
print("\nBy matched status of the storefront (share of its storefront-months):")
display(pd.crosstab(history_check.matched, history_check.status, normalize = "index").round(3))
print("\nStorefronts open by WSLCB in 2019-01 - 2020-07 with no sales record in any of those months:")
no_sales_ever = history_check[history_check.open_wslcb].groupby(STOREFRONT).has_sales.any()
print((~no_sales_ever).sum(), "of", len(no_sales_ever))

## What the match rate would be with the universe gated by sales: drop storefronts open in 2019-01 - 2020-07 with no sales in any of those months
## (storefronts that only open after Jul 2020 can't be checked and stay in)
gate = coverage.merge(no_sales_ever.rename("any_sales").reset_index(), on = STOREFRONT, how = "left")
gate["kept"] = gate.any_sales.fillna(True).astype(bool)
print(f"storefronts kept: {gate.kept.sum()} of {len(gate)} | match rate {gate.matched.mean():.1%} -> {gate[gate.kept].matched.mean():.1%}"
      f" | dropped: {(~gate.kept & gate.matched).sum()} matched, {(~gate.kept & ~gate.matched).sum()} unmatched")
hm = history_check[history_check.open_wslcb]
print(f"storefront-months 2019-01 - 2020-07 covered: all open {hm.covered.mean():.1%}, open with sales {hm[hm.has_sales].covered.mean():.1%}")
print("\nMatched storefronts with no sales in any open month of 2019-01 - 2020-07:")
display(gate[gate.matched & ~gate.kept][STOREFRONT + ["tradenames", "licenses", "open_months", "covered_months"]])

license-months operating at 2+ storefronts at once (sales split equally): 87


,storefront_months,storefronts,share
status,,,
"open, no sales record",1348,168,0.140842
"open, sales",8222,475,0.859053
"sales, not open by WSLCB",1,1,0.000104



By matched status of the storefront (share of its storefront-months):


status,"open, no sales record","open, sales","sales, not open by WSLCB"
matched,,,
False,0.571,0.428,0.001
True,0.048,0.952,0.000



Storefronts open by WSLCB in 2019-01 - 2020-07 with no sales record in any of those months:
58 of 533
storefronts kept: 484 of 542 | match rate 78.8% -> 86.6% | dropped: 8 matched, 50 unmatched
storefront-months 2019-01 - 2020-07 covered: all open 57.1%, open with sales 64.9%

Matched storefronts with no sales in any open month of 2019-01 - 2020-07:


,street_address_norm,city_norm,tradenames,licenses,open_months,covered_months
63,12001 NE 4TH PLAIN BLVD,VANCOUVER,[THE HERBERY],[430925],24,8
109,1411 E OLIVE WAY,SEATTLE,"[NORHTWEST CANNABIS, NORTHWEST CANNABIS, UNCLE...",[413996],36,11
136,15935 REDMOND WAY,REDMOND,[THE GRASS IS ALWAYS GREENER],[430424],24,4
160,17903 HWY 536,MOUNT VERNON,"[CANNAZONE, FWEEDOM CANNABIS]",[422000],24,18
195,20019 AURORA AVE N,SHORELINE,[KUSHKLUB],[430737],24,4
201,203 KINWOOD ST SE,LACEY,[DANK'S WONDER EMPORIUM],[430691],24,6
345,3818 OLD HWY 99 S RD,MOUNT VERNON,"[CANNA CASA, FORBIDDEN CANNABIS CLUB MOUNT VER...",[422913],36,4
521,931 E JOHNS PRAIRIE RD,SHELTON,"[FLOYDS, SASHA'S CANNABIS]",[430798],24,3


In [5]:
### PART A, SUBPART II
## Output: [9aii] licenses with positive retail sales in some month of 2019-01 - 2020-07 when no storefront has them operating by [1e']
##         Columns are "license", "months_with_sales_not_operating", "first_month", "last_month", "sales" (total in those months), "in_1a" (the license is in
##         [1a] at all), "storefronts_in_1a" (its addresses in [1a]), "any_storefront_in_universe" (one of those addresses is one of the 542 storefronts)
## Name: 9aii_sales_without_operating_license.parquet
operating = lic_store[["license", "month"]].drop_duplicates().assign(op = True)
orphan = (s[s.sales > 0].merge(operating, on = ["license", "month"], how = "left"))
orphan = orphan[orphan.op.isna()]
addresses = locations.assign(storefront = locations.street_address_norm + ", " + locations.city_norm).groupby("license").storefront.apply(lambda x: sorted(set(x)))
universe = set(coverage.street_address_norm + ", " + coverage.city_norm)
orphans = (orphan.groupby("license").agg(months_with_sales_not_operating = ("month", "size"), first_month = ("month", "min"),
                                         last_month = ("month", "max"), sales = ("sales", "sum")).reset_index())
orphans["storefronts_in_1a"] = orphans.license.map(addresses)
orphans["in_1a"] = orphans.storefronts_in_1a.notna()
orphans["any_storefront_in_universe"] = orphans.storefronts_in_1a.apply(lambda x: isinstance(x, list) and any(a in universe for a in x))
orphans.to_parquet("processed-data/9aii_sales_without_operating_license.parquet")

total = s[s.sales > 0]
print(f"license-months with sales: {len(total)}; with no storefront operating them by [1e']: {len(orphan)} ({len(orphan) / len(total):.1%}),"
      f" {orphan.sales.sum() / total.sales.sum():.1%} of sales")
print("those licenses:", len(orphans), "| not in [1a] at all:", (~orphans.in_1a).sum(),
      "| in [1a], an address in the 542:", orphans.any_storefront_in_universe.sum(),
      "| in [1a], no address in the 542:", (orphans.in_1a & ~orphans.any_storefront_in_universe).sum())
display(orphans.sort_values("months_with_sales_not_operating", ascending = False).head(15))

license-months with sales: 8250; with no storefront operating them by [1e']: 25 (0.3%), 0.0% of sales
those licenses: 3 | not in [1a] at all: 0 | in [1a], an address in the 542: 3 | in [1a], no address in the 542: 0


,license,months_with_sales_not_operating,first_month,last_month,sales,storefronts_in_1a,in_1a,any_storefront_in_universe
0,413846,14,2019-01,2020-03,91344.22,"[551 14TH AVE, LONGVIEW, 946 WASHINGTON WAY, L...",True,True
1,420800,10,2019-01,2019-10,160046.70,"[141 SAMISH WAY, BELLINGHAM, 2119 LINCOLN ST, ...",True,True
2,421577,1,2020-01,2020-01,310817.62,"[7829 LEARY WAY NE, REDMOND, 8296 AVONDALE WAY...",True,True


In [6]:
### PART A, CHART
monthly = history_check.groupby(["month", "status"]).size().rename("storefronts").reset_index()
fig = px.bar(monthly, x = "month", y = "storefronts", color = "status", title = "Storefronts by month: WSLCB samples vs. LCB sales records (2019-01 to 2020-07)",
             category_orders = {"status": ["open, sales", "open, no sales record", "sales, not open by WSLCB"]})
show(fig, "store_history_vs_sales_by_month")

### PART B: places

In [7]:
### LOADING: place characteristics
## ACS 2017-2021 5-year, by place (summary level 160). "-" and the ACS jam values (e.g. -666666666) mean no estimate
def acs(table, cols):
    d = pd.read_csv(f"{DOWNLOADS}/acsdt5y2021-{table}.dat", sep = "|", dtype = str, usecols = ["GEO_ID"] + cols)
    d = d[d.GEO_ID.str.startswith("1600000US53")].copy()
    d["place_fips"] = d.GEO_ID.str[-7:]
    for c in cols:
        d[c] = pd.to_numeric(d[c], errors = "coerce")
        d.loc[d[c] < 0, c] = np.nan
    return d.drop(columns = "GEO_ID")

acs_place = (acs("b19013", ["B19013_E001"])
    .merge(acs("b23025", ["B23025_E003", "B23025_E005"]), on = "place_fips", how = "outer")
    .merge(acs("b17001", ["B17001_E001", "B17001_E002"]), on = "place_fips", how = "outer"))
acs_place["median_hh_income"] = acs_place.B19013_E001
acs_place["unemployment_rate"] = acs_place.B23025_E005 / acs_place.B23025_E003.where(acs_place.B23025_E003 > 0)
acs_place["poverty_rate"] = acs_place.B17001_E002 / acs_place.B17001_E001.where(acs_place.B17001_E001 > 0)

## Land area: TIGER/Line 2020 places
place_shapes = gpd.read_file(f"zip://{DOWNLOADS}/tl_2020_53_place.zip")
place_area = pd.DataFrame({"place_fips": place_shapes.GEOID, "land_sq_mi": place_shapes.ALAND / 2589988.11})

## Urbanicity: share of the place's 2020 population living in 2020 Census urban blocks
blocks = pl.read_parquet(f"{CENSUS}/p12_blocks_2020.parquet").to_pandas()
urban_blocks = pd.read_csv(f"{DOWNLOADS}/2020_UA_BLOCKS.txt", sep = "|", dtype = str, usecols = ["STATE", "GEOID"])
urban_blocks = set(urban_blocks[urban_blocks.STATE == "53"].GEOID)
blocks["urban"] = blocks.block2020.isin(urban_blocks)
with zipfile.ZipFile(f"{DOWNLOADS}/BlockAssign_ST53_WA.zip") as z:
    block_place = pd.read_csv(z.open("BlockAssign_ST53_WA_INCPLACE_CDP.txt"), sep = "|", dtype = str)
block_place = block_place.dropna(subset = ["PLACEFP"]).assign(place_fips = lambda d: "53" + d.PLACEFP).rename(columns = {"BLOCKID": "block2020"})
urban_place = (blocks.merge(block_place[["block2020", "place_fips"]], on = "block2020")
    .assign(urban_pop = lambda d: d.population_total * d.urban)
    .groupby("place_fips").agg(pop_blocks = ("population_total", "sum"), urban_pop = ("urban_pop", "sum")).reset_index())
urban_place["urban_share"] = urban_place.urban_pop / urban_place.pop_blocks.where(urban_place.pop_blocks > 0)

place_chars = (places[["place_fips", "place_name", "place_type", "population_total"]]
    .merge(place_area, on = "place_fips", how = "left")
    .merge(urban_place[["place_fips", "pop_blocks", "urban_share"]], on = "place_fips", how = "left")
    .merge(acs_place[["place_fips", "median_hh_income", "unemployment_rate", "poverty_rate"]], on = "place_fips", how = "left"))
place_chars["pop_density"] = place_chars.population_total / place_chars.land_sq_mi
print("places:", len(place_chars), "| block sums equal [3d] population:", (place_chars.pop_blocks == place_chars.population_total).mean().round(3))
print("missing:", place_chars[["land_sq_mi", "urban_share", "median_hh_income", "unemployment_rate", "poverty_rate"]].isna().sum().to_dict())

places: 639 | block sums equal [3d] population: 1.0
missing: {'land_sq_mi': 0, 'urban_share': 0, 'median_hh_income': 58, 'unemployment_rate': 13, 'poverty_rate': 5}


In [8]:
### PART B, SUBPART I
## Output: [9bi] one row per census place with at least one of the 542 storefronts
##         Columns are "place_fips", "place_name", "place_type", "storefronts", "matched", "match_rate" (matched / storefronts, the main measure),
##         "open_months", "covered_months", "month_coverage" (covered / open months, an alternative with fewer ties), and the characteristics:
##         "population_total" (2020 Census), "land_sq_mi", "pop_density" (people per square mile), "urban_share" (share of people in 2020 urban blocks),
##         "median_hh_income", "unemployment_rate", "poverty_rate" (ACS 2017-2021)
## Name: 9bi_place_match_rates.parquet
place_rates = (coverage[coverage.place_fips.notna()]
    .groupby("place_fips").agg(storefronts = ("matched", "size"), matched = ("matched", "sum"),
                               open_months = ("open_months", "sum"), covered_months = ("covered_months", "sum")).reset_index())
place_rates["match_rate"] = place_rates.matched / place_rates.storefronts
place_rates["month_coverage"] = place_rates.covered_months / place_rates.open_months
place_rates = place_rates.merge(place_chars, on = "place_fips", how = "left")
place_rates.to_parquet("processed-data/9bi_place_match_rates.parquet")

print("places with a storefront:", len(place_rates), "| storefronts in them:", place_rates.storefronts.sum(),
      "| storefronts outside any place (not in Part B):", coverage.place_fips.isna().sum())
print("places by number of storefronts:", place_rates.storefronts.value_counts().sort_index().to_dict())
print("match rate: share of places at exactly 0:", (place_rates.match_rate == 0).mean().round(3), "| at exactly 1:", (place_rates.match_rate == 1).mean().round(3),
      "| distinct values:", place_rates.match_rate.nunique())

places with a storefront: 170 | storefronts in them: 466 | storefronts outside any place (not in Part B): 76
places by number of storefronts: {1: 78, 2: 41, 3: 21, 4: 7, 5: 10, 6: 5, 7: 3, 12: 1, 15: 1, 16: 2, 55: 1}
match rate: share of places at exactly 0: 0.165 | at exactly 1: 0.6 | distinct values: 13


In [9]:
### PART B, SUBPART II
## Output: [9bii] place characteristics by match-rate group, for terciles, quartiles, and quintiles of the match rate (and of month coverage), all places and
##         places with 3+ storefronts
##         Groups never split tied match rates: each place's group comes from its average rank, so tied places land together and some groups can end up
##         empty or merged (the "groups" column says how many there really are). Group 1 = lowest match rate
##         Columns are "rate", "places_kept", "k", "groups", "group", "places", "storefronts", "rate_min", "rate_max", then the mean and median of each
##         characteristic, and "kruskal_p_<characteristic>" (Kruskal-Wallis test of equal distributions across the groups)
## Name: 9bii_place_characteristics_by_group.parquet
CHARS = {"population_total": "Population (2020)", "pop_density": "People per sq. mile", "urban_share": "Share urban",
         "median_hh_income": "Median household income", "unemployment_rate": "Unemployment rate", "poverty_rate": "Poverty rate"}

def tie_groups(x, k):
    g = np.ceil(x.rank(method = "average") / len(x) * k).clip(1, k).astype(int)
    return g.map({v: i + 1 for i, v in enumerate(sorted(g.unique()))})

rows = []
for rate in ["match_rate", "month_coverage"]:
    for kept, d0 in [("all", place_rates), ("3+ storefronts", place_rates[place_rates.storefronts >= 3])]:
        for k in [3, 4, 5]:
            d = d0.assign(group = tie_groups(d0[rate], k))
            pvals = {f"kruskal_p_{c}": stats.kruskal(*[g[c].dropna() for _, g in d.groupby("group")]).pvalue if d.group.nunique() > 1 else np.nan for c in CHARS}
            for grp, g in d.groupby("group"):
                r = {"rate": rate, "places_kept": kept, "k": k, "groups": d.group.nunique(), "group": grp, "places": len(g), "storefronts": g.storefronts.sum(),
                     "rate_min": g[rate].min(), "rate_max": g[rate].max()}
                for c in CHARS:
                    r[f"mean_{c}"], r[f"median_{c}"] = g[c].mean(), g[c].median()
                rows.append({**r, **pvals})
by_group = pd.DataFrame(rows)
by_group.to_parquet("processed-data/9bii_place_characteristics_by_group.parquet")

main = by_group.query("rate == 'match_rate' and places_kept == 'all' and k == 4")
print("Main version (storefront match rate, all places, quartiles):")
display(main[["group", "places", "storefronts", "rate_min", "rate_max"] + [f"median_{c}" for c in CHARS]].round(3))
display(main[[f"kruskal_p_{c}" for c in CHARS]].iloc[[0]].round(3))

## Spearman correlation of each characteristic with the match rate across places (no grouping, so ties don't matter), weighted equally
corr = []
for rate in ["match_rate", "month_coverage"]:
    for kept, d0 in [("all", place_rates), ("3+ storefronts", place_rates[place_rates.storefronts >= 3])]:
        for c in CHARS:
            ok = d0[[rate, c]].dropna()
            rho, p = stats.spearmanr(ok[rate], ok[c])
            corr.append({"rate": rate, "places_kept": kept, "characteristic": c, "places": len(ok), "spearman_rho": rho, "p_value": p})
corr = pd.DataFrame(corr)
display(corr.pivot_table(index = "characteristic", columns = ["rate", "places_kept"], values = ["spearman_rho", "p_value"]).round(3))

## Storefront level, no grouping: characteristics of the place each matched / unmatched storefront is in (storefronts outside any place left out)
sp = coverage[coverage.place_fips.notna()][["place_fips", "matched"]].merge(place_chars, on = "place_fips", how = "left")
store_level = pd.DataFrame([{"characteristic": c, "matched_median": sp[sp.matched][c].median(), "unmatched_median": sp[~sp.matched][c].median(),
                             "matched_mean": sp[sp.matched][c].mean(), "unmatched_mean": sp[~sp.matched][c].mean(),
                             "ranksum_p": stats.mannwhitneyu(sp[sp.matched][c].dropna(), sp[~sp.matched][c].dropna()).pvalue} for c in CHARS])
print("\nPlace characteristics of matched vs. unmatched storefronts (storefront level, %d storefronts in places):" % len(sp))
display(store_level.round(3))

Main version (storefront match rate, all places, quartiles):


,group,places,storefronts,rate_min,rate_max,median_population_total,median_pop_density,median_urban_share,median_median_hh_income,median_unemployment_rate,median_poverty_rate
3,1,45,77,0.000,0.500,10148.0,1715.967,0.988,73131.0,0.056,0.107
4,2,23,200,0.667,0.875,73256.0,3736.587,1.000,71357.0,0.051,0.114
5,3,102,189,1.000,1.000,7161.0,1478.068,0.989,69028.0,0.051,0.090


,kruskal_p_population_total,kruskal_p_pop_density,kruskal_p_urban_share,kruskal_p_median_hh_income,kruskal_p_unemployment_rate,kruskal_p_poverty_rate
3,0.0,0.0,0.01,0.731,0.579,0.252


p_value                                spearman_rho  \
rate                  match_rate        month_coverage            match_rate   
places_kept       3+ storefronts    all 3+ storefronts    all 3+ storefronts   
characteristic                                                                 
median_hh_income           0.481  0.426          0.969  0.686         -0.101   
pop_density                0.178  0.114          0.379  0.460         -0.192   
population_total           0.337  0.012          0.574  0.970         -0.137   
poverty_rate               0.987  0.392          0.271  0.394         -0.002   
unemployment_rate          0.493  0.391          0.156  0.451         -0.098   
urban_share                0.783  0.744          0.921  0.134         -0.039   

                                                
rate                     month_coverage         
places_kept          all 3+ storefronts    all  
characteristic                                  
median_hh_income  -0.062         -0.006  0.032  
pop_density       -0.122         -0.126  0.057  
population_total  -0.193         -0.081  0.003  
poverty_rate      -0.066         -0.157 -0.066  
unemployment_rate -0.067         -0.202 -0.059  
urban_share       -0.025          0.014  0.116


Place characteristics of matched vs. unmatched storefronts (storefront level, 466 storefronts in places):


,characteristic,matched_median,unmatched_median,matched_mean,unmatched_mean,ranksum_p
0,population_total,27729.000,23569.000,136818.512,116785.697,0.408
1,pop_density,3250.850,3111.961,3585.612,3386.564,0.574
2,urban_share,0.999,0.999,0.905,0.795,0.469
3,median_hh_income,71357.000,72969.000,77872.728,80685.186,0.491
4,unemployment_rate,0.050,0.052,0.055,0.054,0.671
5,poverty_rate,0.106,0.107,0.118,0.122,0.947


In [10]:
### PART B, CHART: medians by match-rate quartile
long = main.melt(id_vars = ["group", "rate_min", "rate_max", "places"], value_vars = [f"median_{c}" for c in CHARS], var_name = "c", value_name = "median")
long["characteristic"] = long.c.str.replace("median_", "").map(CHARS)
long["group_label"] = long.apply(lambda r: f"G{r.group}: {r.rate_min:.0%}-{r.rate_max:.0%} (n={r.places})", axis = 1)
fig = px.bar(long, x = "group_label", y = "median", facet_col = "characteristic", facet_col_wrap = 3, title = "Place characteristics (median) by storefront match-rate group",
             labels = {"group_label": "match-rate group", "median": ""})
fig.update_yaxes(matches = None, showticklabels = True)
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
show(fig, "place_characteristics_by_match_rate")

### PART C: stores

In [11]:
### PART C, SUBPART I
## Output: [9ci] one row per storefront (the 542 in [6a]) with:
##   matching: "matched", "reason_missing", "covered_months", "open_months", "share_months_covered" (from [6a])
##   cohort: "first_sales_month" (earliest month with positive retail sales by any license operating at the storefront, Jul 2014 on), "cohort" (its year;
##           "2014-2015" grouped, "no sales record"), "cohort_uncertain" (the earliest license also operated at another address, so its first sales may be from
##           there), "first_wslcb_operating" (first dated WSLCB sample with a license operating there; the samples start in Jul 2018)
##   months active: "first_open_month", "last_open_month" (2019-2021, [1e']), "gap_months" (months between the first and last open month that aren't open),
##           "spells" (separate runs of open months), "opened_in_window" / "closed_in_window" (first open month after 2019-01 / last before 2021-12)
##   sales (2019-01 - 2020-07, open months): "sales_months" (open months in that range), "mean_monthly_sales", "share_open_months_with_sales",
##           "sales_gap_months" (open months with no sales between the first and last month with sales; [1e'] itself has almost no gaps, since a license
##           counts as operating in every month between two samples that list it)
##   address: "has_suite" (a unit / suite / building in the WSLCB address), "highway_address" (state route, highway, or milepost), "licenses_ever" (licenses
##           at the address in [1a]), "tradenames_ever", "census_geocode" (the Census geocoder's result for the WSLCB address: exact, non-exact, no match;
##           independent of SafeGraph)
##   surroundings (SafeGraph POIs, current snapshot, around the Census geocoder point, or the [1d] point when the geocoder failed): "pois_50m", "pois_150m",
##           "retail_pois_150m" (NAICS 44-45), "food_pois_150m" (NAICS 722)
##   neighborhood: "place_type", "place_size" (from [6a]), "cbg_pop_density" (2020 block group, people per square mile), "cbg_urban_share"
## Name: 9ci_storefront_characteristics.parquet
stores = coverage[STOREFRONT + ["tradenames", "licenses", "matched", "reason_missing", "open_months", "covered_months", "share_months_covered",
                                "latitude", "longitude", "geo_method", "county_name", "place_fips", "place_name", "place_type", "place_size"]].copy()

# (1) Cohort: first positive sales month of the licenses operating at the storefront
lic_first = sales[sales.sales > 0].groupby("license").month.min().rename("first_sales")
st_lic = store_licenses.merge(lic_first, on = "license", how = "left").merge(lic_n_stores, on = "license")
cohort = (st_lic.sort_values("first_sales")
    .groupby(STOREFRONT).agg(first_sales_month = ("first_sales", "first"), first_license_moves = ("n_storefronts_ever", "first")).reset_index())
cohort["cohort_uncertain"] = cohort.first_license_moves > 1
stores = stores.merge(cohort.drop(columns = "first_license_moves"), on = STOREFRONT, how = "left")
yr = stores.first_sales_month.str[:4]
stores = stores.merge(active_first := open_months.groupby(STOREFRONT).month.min().rename("_first_open").reset_index(), on = STOREFRONT, how = "left")
stores["cohort"] = np.where(yr.notna(), np.where(yr.isin(["2014", "2015"]), "2014-2015", yr),
                            np.where(stores._first_open > SALES_MONTHS[-1], "opened after Jul 2020 (no sales data)", "no sales record"))
stores = stores.drop(columns = "_first_open")
first_wslcb = locations.groupby(STOREFRONT).first_operating.min().rename("first_wslcb_operating").reset_index()
stores = stores.merge(first_wslcb, on = STOREFRONT, how = "left")

# (2) Months active and gaps, 2019-2021
sm = open_months[STOREFRONT + ["month"]].assign(i = lambda d: d.month.map(month_index)).sort_values(STOREFRONT + ["i"])
sm["new_spell"] = sm.groupby(STOREFRONT).i.diff().fillna(99) != 1
active = sm.groupby(STOREFRONT).agg(first_open_month = ("month", "min"), last_open_month = ("month", "max"), n = ("i", "size"),
                                    span = ("i", lambda x: x.max() - x.min() + 1), spells = ("new_spell", "sum")).reset_index()
active["gap_months"] = active.span - active.n
active["opened_in_window"] = active.first_open_month > "2019-01"
active["closed_in_window"] = active.last_open_month < "2021-12"
stores = stores.merge(active.drop(columns = ["n", "span"]), on = STOREFRONT, how = "left")

# (3) Sales over the storefront's open months in 2019-01 - 2020-07
def sales_gaps(g):  # open months with no sales record between the storefront's first and last month with sales (2019-01 - 2020-07)
    g = g.sort_values("month")
    idx = np.flatnonzero(g.has_sales.values)
    return 0 if len(idx) == 0 else int((~g.has_sales.values[idx[0]:idx[-1] + 1]).sum())
hc_open = history_check[history_check.open_wslcb]
sales_st = hc_open.groupby(STOREFRONT).agg(
    sales_months = ("month", "size"), mean_monthly_sales = ("sales", lambda x: x.fillna(0).mean()),
    share_open_months_with_sales = ("has_sales", "mean")).reset_index()
sales_st = sales_st.merge(hc_open.groupby(STOREFRONT)[["month", "has_sales"]].apply(sales_gaps).rename("sales_gap_months").reset_index(), on = STOREFRONT)
stores = stores.merge(sales_st, on = STOREFRONT, how = "left")

# (4) Address: suite and highway flags from the WSLCB street address, licenses and tradenames at the address, the Census geocoder's result
addr = locations.groupby(STOREFRONT).agg(raw_addresses = ("street_address", lambda x: sorted(set(x))), licenses_ever = ("license", "nunique"),
                                         tradenames_ever = ("tradenames", lambda x: len(set(t for l in x for t in l))), zip5 = ("zip5", "first")).reset_index()
SUITE = r"\b(STE|SUITE|UNIT|BLDG|BUILDING|APT|SPC|SPACE|RM|ROOM|LOT)\b|#|\b\d+[A-Z]?\s*&\s*[A-Z\d]"
HIGHWAY = r"\b(HWY|HIGHWAY|SR|STATE ROUTE|ROUTE|RTE|US|MP|MILEPOST|MILE POST)\b|\bWA-?\d"
addr["has_suite"] = addr.raw_addresses.apply(lambda a: any(re.search(SUITE, x.upper()) for x in a))
addr["highway_address"] = addr.street_address_norm.str.upper().str.contains(HIGHWAY, regex = True)
stores = stores.merge(addr.drop(columns = "raw_addresses"), on = STOREFRONT, how = "left")

geo = pd.read_csv(f"{CENSUS}/wslcb_census_geocoder.csv", header = None, dtype = str,
                  names = ["geocode_id", "input", "match", "match_type", "matched_address", "lonlat", "tiger_id", "side", "state", "county", "tract", "block"])
geo["input"] = geo.input.str.upper().str.strip()
inputs = locations.assign(input = (locations.street_address + ", " + locations.city_norm + ", WA, " + locations.zip5.fillna("")).str.upper().str.strip())
geo = inputs[STOREFRONT + ["input"]].merge(geo, on = "input", how = "inner")
geo["census_geocode"] = np.where(geo.match == "Match", np.where(geo.match_type == "Exact", "exact", "non-exact"), "no match")
geo[["glon", "glat"]] = geo.lonlat.str.split(",", expand = True).astype(float)
rank = {"exact": 0, "non-exact": 1, "no match": 2}
geo = geo.assign(r = geo.census_geocode.map(rank)).sort_values("r").drop_duplicates(STOREFRONT)
stores = stores.merge(geo[STOREFRONT + ["census_geocode", "glat", "glon"]], on = STOREFRONT, how = "left")
print("storefronts with a Census geocoder row:", stores.census_geocode.notna().sum(), "of", len(stores))

# (5) Surrounding businesses: SafeGraph POIs within 50 m / 150 m of the Census geocoder point (or the [1d] point when the geocoder has none)
stores["pt_lat"] = stores.glat.where(stores.glat.notna(), stores.latitude)
stores["pt_lon"] = stores.glon.where(stores.glon.notna(), stores.longitude)
stores["point_source"] = np.where(stores.glat.notna(), "census geocoder", "[1d]: " + stores.geo_method.astype(str))
poi = (pl.scan_parquet(f"{POI_FOLDER}/*.parquet").filter(pl.col("REGION") == "WA")
       .select(["PLACEKEY", "LATITUDE", "LONGITUDE", "NAICS_CODE"]).collect().to_pandas())
poi = gpd.GeoDataFrame(poi, geometry = gpd.points_from_xy(poi.LONGITUDE, poi.LATITUDE), crs = 4326).to_crs(32610)
pts = gpd.GeoDataFrame(stores[STOREFRONT], geometry = gpd.points_from_xy(stores.pt_lon, stores.pt_lat), crs = 4326).to_crs(32610)
for r in [50, 150]:
    j = gpd.sjoin(poi, gpd.GeoDataFrame(pts[STOREFRONT], geometry = pts.buffer(r)), predicate = "within")
    j["naics2"] = j.NAICS_CODE.astype(str).str[:2]
    j["naics3"] = j.NAICS_CODE.astype(str).str[:3]
    agg = j.groupby(STOREFRONT).agg(n = ("PLACEKEY", "size"), retail = ("naics2", lambda x: x.isin(["44", "45"]).sum()),
                                    food = ("naics3", lambda x: (x == "722").sum())).reset_index()
    agg = agg.rename(columns = {"n": f"pois_{r}m", "retail": f"retail_pois_{r}m", "food": f"food_pois_{r}m"})
    stores = stores.merge(agg, on = STOREFRONT, how = "left")
    stores[[f"pois_{r}m", f"retail_pois_{r}m", f"food_pois_{r}m"]] = stores[[f"pois_{r}m", f"retail_pois_{r}m", f"food_pois_{r}m"]].fillna(0)
print("WA SafeGraph POIs:", len(poi))

# (6) Neighborhood: the 2020 block group the [1d] point is in ([1d] cbg2020): density and urban share
bg = gpd.read_file(f"zip://{DOWNLOADS}/tl_2020_53_bg.zip")
bg = pd.DataFrame({"cbg2020": bg.GEOID, "land_sq_mi": bg.ALAND / 2589988.11})
bg_pop = blocks.assign(cbg2020 = blocks.block2020.str[:12], urban_pop = blocks.population_total * blocks.urban).groupby("cbg2020")[["population_total", "urban_pop"]].sum().reset_index()
bg = bg.merge(bg_pop, on = "cbg2020", how = "left")
bg["cbg_pop_density"] = bg.population_total / bg.land_sq_mi.where(bg.land_sq_mi > 0)
bg["cbg_urban_share"] = bg.urban_pop / bg.population_total.where(bg.population_total > 0)
store_cbg = coords.groupby(STOREFRONT).cbg2020.first().reset_index()
stores = stores.merge(store_cbg, on = STOREFRONT, how = "left").merge(bg[["cbg2020", "cbg_pop_density", "cbg_urban_share"]], on = "cbg2020", how = "left")

stores.drop(columns = ["glat", "glon"]).to_parquet("processed-data/9ci_storefront_characteristics.parquet")
print("storefronts:", len(stores), "| cohort uncertain:", stores.cohort_uncertain.sum(), "| no sales record:", (stores.cohort == "no sales record").sum())
display(stores.groupby("matched").point_source.value_counts().unstack(0))

storefronts with a Census geocoder row: 541 of 542


WA SafeGraph POIs: 568297


storefronts: 542 | cohort uncertain: 15 | no sales record: 47


matched,False,True
point_source,,
[1d]: matched_placekey,1.0,2.0
[1d]: openstreetmap,2.0,NaN
[1d]: same_address_placekey,NaN,8.0
[1d]: same_address_poi,8.0,24.0
[1d]: zcta,3.0,2.0
census geocoder,101.0,391.0


In [12]:
### PART C, SUBPART II
## Output: [9cii] matched vs. unmatched storefronts, one row per characteristic: "characteristic", "kind" (share or continuous), "matched_n", "unmatched_n",
##         "matched_mean", "unmatched_mean", "difference" (matched - unmatched), "matched_median", "unmatched_median", "p_value" (Welch t-test for continuous,
##         chi-square / Fisher for shares, rank-sum shown separately as "ranksum_p"), and the means by reason the store is missing
## Name: 9cii_matched_vs_unmatched_storefronts.parquet
d = stores.copy()
d["log_mean_monthly_sales"] = np.log1p(d.mean_monthly_sales)
COHORTS = ["2014-2015", "2016", "2017", "2018", "2019", "2020", "no sales record", "opened after Jul 2020 (no sales data)"]
for c in COHORTS:
    d[f"cohort_{c}"] = d.cohort == c
d["census_geocode_exact"] = d.census_geocode == "exact"
d["census_geocode_failed"] = d.census_geocode == "no match"
for t in ["city", "CDP", "outside any place"]:
    d[f"place_type_{t}"] = d.place_type.fillna("outside any place").replace({"town": "city"}) == t

SHARES = ([f"cohort_{c}" for c in COHORTS] + ["cohort_uncertain",
          "opened_in_window", "closed_in_window", "has_suite", "highway_address", "census_geocode_exact", "census_geocode_failed",
          "place_type_city", "place_type_CDP", "place_type_outside any place"])
CONT = ["open_months", "gap_months", "spells", "sales_gap_months", "mean_monthly_sales", "log_mean_monthly_sales", "share_open_months_with_sales", "licenses_ever",
        "tradenames_ever", "pois_50m", "pois_150m", "retail_pois_150m", "food_pois_150m", "cbg_pop_density", "cbg_urban_share"]
rows = []
m, u = d[d.matched], d[~d.matched]
for c in SHARES + CONT:
    a, b = m[c].dropna().astype(float), u[c].dropna().astype(float)
    if c in SHARES:
        tab = np.array([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]])
        p = stats.fisher_exact(tab).pvalue if tab.min() < 5 else stats.chi2_contingency(tab)[1]
    else:
        p = stats.ttest_ind(a, b, equal_var = False).pvalue
    r = {"characteristic": c, "kind": "share" if c in SHARES else "continuous", "matched_n": len(a), "unmatched_n": len(b),
         "matched_mean": a.mean(), "unmatched_mean": b.mean(), "difference": a.mean() - b.mean(),
         "matched_median": a.median(), "unmatched_median": b.median(), "p_value": p, "ranksum_p": stats.mannwhitneyu(a, b).pvalue}
    for reason, g in u.groupby("reason_missing"):
        r[f"mean_if_{reason}"] = g[c].dropna().astype(float).mean()
    rows.append(r)
compare = pd.DataFrame(rows)
compare.to_parquet("processed-data/9cii_matched_vs_unmatched_storefronts.parquet")
pd.set_option("display.width", 250)
display(compare[["characteristic", "matched_n", "unmatched_n", "matched_mean", "unmatched_mean", "difference", "matched_median", "unmatched_median", "p_value"]].round(3))
print("Unmatched storefronts by reason:", u.reason_missing.value_counts().to_dict())
display(compare[["characteristic"] + [c for c in compare.columns if c.startswith("mean_if_")]].round(3))

## All together: linear probability model of matched on the characteristics, county fixed effects, robust standard errors
reg = d.assign(cohort_c = d.cohort.replace({"no sales record": "none", "opened after Jul 2020 (no sales data)": "after Jul 2020"}), place_t = d.place_type.fillna("outside").replace({"town": "city", "outside any place": "outside"}),
               log_pois_150m = np.log1p(d.pois_150m), log_cbg_density = np.log1p(d.cbg_pop_density), matched_i = d.matched.astype(int),
               has_suite_i = d.has_suite.astype(int), highway_i = d.highway_address.astype(int), exact_i = d.census_geocode_exact.astype(int))
# gap_months is left out: [1e'] has almost no gaps (a handful of storefronts), so its coefficient would rest on those few
formula = ("matched_i ~ C(cohort_c, Treatment('2014-2015')) + open_months + has_suite_i + highway_i + exact_i + licenses_ever"
           " + log_pois_150m + log_cbg_density + C(place_t, Treatment('city'))")
for label, f, data in [("without sales (all storefronts)", formula, reg),
                       ("with sales (2019-01 - 2020-07 open months)", formula + " + log_mean_monthly_sales", reg[reg.sales_months > 0])]:
    for fe in ["", " + C(county_name)"]:
        fit = smf.ols(f + fe, data = data.dropna(subset = ["log_cbg_density"])).fit(cov_type = "HC1")
        print(f"\n=== LPM {label}{', county FE' if fe else ''}: n = {int(fit.nobs)}, R2 = {fit.rsquared:.3f}")
        out = pd.DataFrame({"coef": fit.params, "se": fit.bse, "p": fit.pvalues})
        display(out[~out.index.str.startswith("C(county_name)")].round(3))

,characteristic,matched_n,unmatched_n,matched_mean,unmatched_mean,difference,matched_median,unmatched_median,p_value
0,cohort_2014-2015,427,115,0.363,0.104,0.259,0.000,0.000,0.000
1,cohort_2016,427,115,0.286,0.070,0.216,0.000,0.000,0.000
2,cohort_2017,427,115,0.159,0.087,0.072,0.000,0.000,0.070
3,cohort_2018,427,115,0.105,0.183,-0.077,0.000,0.000,0.037
4,cohort_2019,427,115,0.042,0.070,-0.027,0.000,0.000,0.330
5,cohort_2020,427,115,0.033,0.043,-0.011,0.000,0.000,0.789
6,cohort_no sales record,427,115,0.009,0.374,-0.365,0.000,0.000,0.000
7,cohort_opened after Jul 2020 (no sales data),427,115,0.002,0.070,-0.067,0.000,0.000,0.000
8,cohort_uncertain,427,115,0.014,0.078,-0.064,0.000,0.000,0.001
9,opened_in_window,427,115,0.037,0.165,-0.128,0.000,0.000,0.000


Unmatched storefronts by reason: {'no Veraset place at the address': 63, 'only other businesses at the address': 38, 'name only outside its WSLCB months': 10, 'rejected in review': 4}


,characteristic,mean_if_name only outside its WSLCB months,mean_if_no Veraset place at the address,mean_if_only other businesses at the address,mean_if_rejected in review
0,cohort_2014-2015,0.100,0.095,0.105,0.250
1,cohort_2016,0.300,0.048,0.053,0.000
2,cohort_2017,0.200,0.048,0.105,0.250
3,cohort_2018,0.300,0.190,0.132,0.250
4,cohort_2019,0.100,0.079,0.026,0.250
5,cohort_2020,0.000,0.032,0.079,0.000
6,cohort_no sales record,0.000,0.460,0.368,0.000
7,cohort_opened after Jul 2020 (no sales data),0.000,0.048,0.132,0.000
8,cohort_uncertain,0.400,0.048,0.053,0.000
9,opened_in_window,0.000,0.143,0.263,0.000



=== LPM without sales (all storefronts): n = 542, R2 = 0.499


,coef,se,p
Intercept,0.074,0.124,0.554
"C(cohort_c, Treatment('2014-2015'))[T.2016]",0.011,0.026,0.679
"C(cohort_c, Treatment('2014-2015'))[T.2017]",-0.051,0.038,0.182
"C(cohort_c, Treatment('2014-2015'))[T.2018]",-0.214,0.055,0.000
"C(cohort_c, Treatment('2014-2015'))[T.2019]",-0.243,0.090,0.007
"C(cohort_c, Treatment('2014-2015'))[T.2020]",-0.044,0.110,0.686
"C(cohort_c, Treatment('2014-2015'))[T.after Jul 2020]",-0.408,0.121,0.001
"C(cohort_c, Treatment('2014-2015'))[T.none]",-0.670,0.062,0.000
"C(place_t, Treatment('city'))[T.CDP]",-0.006,0.033,0.848
"C(place_t, Treatment('city'))[T.outside]",0.067,0.048,0.161



=== LPM without sales (all storefronts), county FE: n = 542, R2 = 0.539


,coef,se,p
Intercept,-0.114,0.206,0.581
"C(cohort_c, Treatment('2014-2015'))[T.2016]",0.012,0.028,0.662
"C(cohort_c, Treatment('2014-2015'))[T.2017]",-0.037,0.041,0.364
"C(cohort_c, Treatment('2014-2015'))[T.2018]",-0.217,0.057,0.000
"C(cohort_c, Treatment('2014-2015'))[T.2019]",-0.233,0.091,0.011
"C(cohort_c, Treatment('2014-2015'))[T.2020]",-0.032,0.105,0.760
"C(cohort_c, Treatment('2014-2015'))[T.after Jul 2020]",-0.426,0.128,0.001
"C(cohort_c, Treatment('2014-2015'))[T.none]",-0.677,0.067,0.000
"C(place_t, Treatment('city'))[T.CDP]",-0.025,0.034,0.472
"C(place_t, Treatment('city'))[T.outside]",0.066,0.048,0.167



=== LPM with sales (2019-01 - 2020-07 open months): n = 533, R2 = 0.521


,coef,se,p
Intercept,-0.195,0.115,0.089
"C(cohort_c, Treatment('2014-2015'))[T.2016]",0.014,0.024,0.545
"C(cohort_c, Treatment('2014-2015'))[T.2017]",-0.042,0.035,0.224
"C(cohort_c, Treatment('2014-2015'))[T.2018]",-0.178,0.055,0.001
"C(cohort_c, Treatment('2014-2015'))[T.2019]",-0.180,0.085,0.034
"C(cohort_c, Treatment('2014-2015'))[T.2020]",0.090,0.125,0.470
"C(cohort_c, Treatment('2014-2015'))[T.none]",-0.190,0.101,0.059
"C(place_t, Treatment('city'))[T.CDP]",0.007,0.031,0.835
"C(place_t, Treatment('city'))[T.outside]",0.070,0.045,0.124
open_months,0.014,0.003,0.000



=== LPM with sales (2019-01 - 2020-07 open months), county FE: n = 533, R2 = 0.561


,coef,se,p
Intercept,-0.376,0.199,0.058
"C(cohort_c, Treatment('2014-2015'))[T.2016]",0.016,0.026,0.531
"C(cohort_c, Treatment('2014-2015'))[T.2017]",-0.027,0.039,0.493
"C(cohort_c, Treatment('2014-2015'))[T.2018]",-0.179,0.055,0.001
"C(cohort_c, Treatment('2014-2015'))[T.2019]",-0.155,0.085,0.067
"C(cohort_c, Treatment('2014-2015'))[T.2020]",0.116,0.121,0.339
"C(cohort_c, Treatment('2014-2015'))[T.none]",-0.167,0.107,0.118
"C(place_t, Treatment('city'))[T.CDP]",-0.001,0.032,0.984
"C(place_t, Treatment('city'))[T.outside]",0.071,0.047,0.129
open_months,0.013,0.003,0.000


In [13]:
### PART C, CHARTS
coh = d.groupby(["matched", "cohort"]).size().rename("storefronts").reset_index()
coh["share"] = coh.storefronts / coh.groupby("matched").storefronts.transform("sum")
coh["group"] = np.where(coh.matched, "matched", "unmatched")
fig = px.bar(coh, x = "cohort", y = "share", color = "group", barmode = "group", title = "Opening cohort (first month with LCB retail sales), matched vs. unmatched storefronts",
             category_orders = {"cohort": COHORTS})
show(fig, "store_cohort_matched_vs_unmatched")

box = d.melt(id_vars = ["matched"], value_vars = ["mean_monthly_sales", "open_months", "pois_150m", "cbg_pop_density"], var_name = "characteristic")
box["group"] = np.where(box.matched, "matched", "unmatched")
fig = px.box(box, x = "group", y = "value", facet_col = "characteristic", points = "outliers", title = "Matched vs. unmatched storefronts")
fig.update_yaxes(matches = None, showticklabels = True)
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
show(fig, "store_characteristics_matched_vs_unmatched")

In [14]:
### PART C, SUBPART III
## Output: [9ciii] one row per open storefront-month in 2019-2021 ([1e'], the [6a] universe) with "covered" (matched that month), "matched_store",
##         "months_since_first_sales" (from the storefront's first LCB sales month), "sales" and "has_sales" (that month, 2019-01 - 2020-07 only), "year",
##         "edge_month" (within the first or last 3 open months of the storefront's 2019-2021 run), and the storefront characteristics from [9ci]
## Name: 9ciii_storefront_months.parquet
smo = (store_months[STOREFRONT + ["month", "covered"]]
    .merge(history_check[history_check.open_wslcb][STOREFRONT + ["month", "sales", "has_sales"]], on = STOREFRONT + ["month"], how = "left")
    .merge(d[STOREFRONT + ["matched", "first_sales_month", "first_open_month", "last_open_month", "cohort", "has_suite", "highway_address",
                           "pois_150m", "cbg_pop_density", "place_type"]], on = STOREFRONT, how = "left")
    .rename(columns = {"matched": "matched_store"}))
smo["i"] = smo.month.map(month_index)
smo["months_since_first_sales"] = smo.i - smo.first_sales_month.map(lambda x: month_index(x) if isinstance(x, str) else np.nan)
smo["edge_month"] = (smo.i - smo.first_open_month.map(month_index) < 3) | (smo.last_open_month.map(month_index) - smo.i < 3)
smo["year"] = smo.month.str[:4]
smo["in_sales_window"] = smo.month.isin(SALES_MONTHS)
smo.to_parquet("processed-data/9ciii_storefront_months.parquet")

print("storefront-months:", len(smo), "| covered:", smo.covered.mean().round(3))
print("\nCoverage by year:"); display(smo.groupby("year").covered.mean().round(3))
w = smo[smo.in_sales_window]
print("\nIn 2019-01 - 2020-07: covered vs. not, all storefront-months")
display(w.groupby("covered").agg(storefront_months = ("month", "size"), share_with_sales = ("has_sales", "mean"),
                                 mean_sales = ("sales", lambda x: x.fillna(0).mean()), median_sales_if_any = ("sales", lambda x: x[x > 0].median()),
                                 edge_month = ("edge_month", "mean"), months_since_first_sales = ("months_since_first_sales", "median")).round(3))

## Within-store: matched storefronts with both covered and uncovered months in the sales window, mean sales in each
ws = w[w.matched_store].groupby(STOREFRONT + ["covered"]).sales.apply(lambda x: x.fillna(0).mean()).unstack("covered").dropna()
ws.columns = ["uncovered_mean_sales", "covered_mean_sales"]
print(f"\nWithin-store (storefronts with both kinds of months: {len(ws)}): median covered/uncovered sales ratio",
      round((ws.covered_mean_sales / ws.uncovered_mean_sales.where(ws.uncovered_mean_sales > 0)).median(), 3),
      "| Wilcoxon signed-rank p =", round(stats.wilcoxon(ws.covered_mean_sales, ws.uncovered_mean_sales).pvalue, 4))

## Storefront fixed effects: does a month's sales predict coverage within a storefront? Month fixed effects, clustered by storefront
fe = w[w.matched_store & w.sales.notna()].copy()
fe["log_sales"] = np.log1p(fe.sales.clip(lower = 0))
fe["sid"] = fe.street_address_norm + "|" + fe.city_norm
fe["covered_i"] = fe.covered.astype(int)
fe["edge_i"] = fe.edge_month.astype(int)
fit = smf.ols("covered_i ~ log_sales + edge_i + C(month) + C(sid)", data = fe).fit(cov_type = "cluster", cov_kwds = {"groups": pd.factorize(fe.sid)[0]})
print(f"\nStorefront + month FE, matched storefronts, n = {int(fit.nobs)}:")
display(pd.DataFrame({"coef": fit.params, "se": fit.bse, "p": fit.pvalues}).loc[["log_sales", "edge_i"]].round(4))

## Coverage by sales decile (matched storefronts, sales window)
fe["sales_decile"] = pd.qcut(fe.sales.rank(method = "first"), 10, labels = range(1, 11)).astype(int)
dec = fe.groupby("sales_decile").agg(coverage = ("covered", "mean"), median_sales = ("sales", "median")).reset_index()
fig = px.line(dec, x = "sales_decile", y = "coverage", markers = True, hover_data = ["median_sales"],
              title = "Share of storefront-months matched, by that month's sales decile (matched storefronts, 2019-01 to 2020-07)")
show(fig, "store_month_coverage_by_sales_decile")

storefront-months: 17742 | covered: 0.617

Coverage by year:


year
2019    0.533
2020    0.649
2021    0.674
Name: covered, dtype: float64


In 2019-01 - 2020-07: covered vs. not, all storefront-months


,storefront_months,share_with_sales,mean_sales,median_sales_if_any,edge_month,months_since_first_sales
covered,,,,,,
False,4103,0.702413,137497.635,148978.785,0.211,34.0
True,5467,0.97677,247266.194,186722.150,0.153,42.0



Within-store (storefronts with both kinds of months: 171): median covered/uncovered sales ratio 1.083 | Wilcoxon signed-rank p = 0.0



Storefront + month FE, matched storefronts, n = 7494:


,coef,se,p
log_sales,0.0082,0.0187,0.6619
edge_i,-0.0002,0.0064,0.9775


In [15]:
### PART C, SUBPART IV
## Output: [9civ] a list to check storefront size by hand: 10 matched and 10 unmatched storefronts (random, seed 9), with the address and a Google Maps link.
##         Columns to fill in by hand: "storefront_size" (small / medium / large / can't tell), "standalone_or_in_strip", "notes"
## Name: review/9c_storefront_size_checklist.csv
pick = pd.concat([d[d.matched].sample(10, random_state = 9), d[~d.matched].sample(10, random_state = 9)])
pick = pick[STOREFRONT + ["zip5", "tradenames", "matched", "reason_missing", "county_name", "pois_50m"]].copy()
pick["tradenames"] = pick.tradenames.apply(lambda x: "; ".join(x) if isinstance(x, (list, np.ndarray)) else x)
pick["google_maps"] = "https://www.google.com/maps/search/?api=1&query=" + (pick.street_address_norm + " " + pick.city_norm + " WA " + pick.zip5.fillna("")).str.replace(" ", "+")
pick[["storefront_size", "standalone_or_in_strip", "notes"]] = ""
pick.to_csv("review/9c_storefront_size_checklist.csv", index = False)
display(pick[STOREFRONT + ["tradenames", "matched", "reason_missing"]])

,street_address_norm,city_norm,tradenames,matched,reason_missing
293,3010 OLD HWY 99 S RD,MOUNT VERNON,CANNAZONE,True,NaN
71,1225 HOLLIS ST,BREMERTON,DESTINATION HIGHWAY 420,True,NaN
45,11401 RAINIER AVE S,SEATTLE,CLOUD CANNABIS; WESTERN BUD,True,NaN
155,17517 15TH AVE NE,SHORELINE,365 RECREATIONAL CANNABIS,True,NaN
441,613 MONTANA AVE,SOUTH BEND,GROWERS OUTLET,True,NaN
506,8630 S MARCH POINT RD,ANACORTES,HIGH SOCIETY; HIGH SOCIETY ANACORTES,True,NaN
375,420 SUNSET BLVD N,RENTON,BUDDY'S,True,NaN
206,20925 CYPRESS WAY,LYNNWOOD,EUPHORIUM 420,True,NaN
117,145 SAMISH WAY,BELLINGHAM,STARBUDS,True,NaN
119,14608 HWY 99,LYNNWOOD,A GREENER TODAY MARIJUANA-LYNNWOOD; PUGET SOUN...,True,NaN


### NOTES
(Written from the outputs above as of the last run. If the inputs change, re-check the numbers.)

**Store history vs. sales (Part A)**
- The task's `store_history_2018_2021.zip` isn't here yet, so Part A checks the [1e'] store history against the LCB's archived monthly sales by license
  instead. It should be rerun with the store history file (and Henry is gating by sales next, so this is a first look, not the final gate).
- 86% of open storefront-months in 2019-01 - 2020-07 have positive sales by a license operating there; 14% (1,348 storefront-months, 168 storefronts) have no
  sales record. **58 of the 533 storefronts open in that window have no sales in any month of it**, and no other license ever at those addresses has sales
  either, so they look like licensed but not (yet) operating storefronts.
- The other direction is clean: only 3 licenses (25 license-months, ~0% of sales) sell in a month when [1e'] doesn't have them operating anywhere. The WSLCB
  samples don't miss operating stores; if anything they include too many.
- Missing sales are concentrated in the unmatched storefronts: 57% of their open months have no sales record, against 5% for matched storefronts.
- Gating the universe by sales (dropping the 58) would take it from 542 to 484 storefronts (8 matched, 50 unmatched dropped) and the **storefront match rate
  from 78.8% to 86.6%**. Storefront-month coverage in 2019-01 - 2020-07 goes from 57.1% to 64.9%.
- The 8 matched storefronts with no sales are mostly newer licenses (430xxx) with 24 open months but only 3-18 covered months, which suggests they opened
  late and the "listed operating in the sample before or after" rule gives them months before they opened.
- 87 license-months have one license operating at two storefronts at once (a move between samples); their sales are split equally.

**Places (Part B)**
- 170 places hold 466 of the 542 storefronts (76 are outside any place). Most places are small: 78 have one storefront, so their match rate is 0 or 1.
  60% of places are at exactly 100% and 17% at 0%, with only 13 distinct values.
- **Quartiles can't be formed without splitting ties**: every version (terciles, quartiles, quintiles) collapses to 3 groups, with the 102 places at 100% in
  the top group. Month coverage (covered / open months) has far fewer ties and gives proper quartiles, so it's the better robustness check.
- **Income, unemployment, and poverty don't differ across match-rate groups** (Kruskal-Wallis p = 0.73, 0.58, 0.25 for match-rate quartiles; 0.93, 0.73,
  0.75 for month-coverage quartiles). Urban share differs a little in the match-rate groups (p = 0.01) but not by month coverage (p = 0.20).
- Population and density do differ, but mechanically: places with many storefronts (the big cities) can hardly be at exactly 0 or 1, so they fill the
  middle group (median population 73k against 7-10k). By month coverage the gap is smaller (p = 0.04), and no characteristic is correlated with the match
  rate among places with 3+ storefronts (all Spearman p > 0.15).
- At the storefront level, which avoids the ties, matched and unmatched storefronts sit in places that look the same (all six characteristics, rank-sum
  p > 0.4).

**Stores (Part C)**
- **Opening cohort**: matched storefronts are older. 36% of matched first sold in 2014-2015 against 10% of unmatched, and the 2018 cohort is 10% against 18%.
  37% of unmatched storefronts have no LCB sales record at all (Jul 2014 - Jul 2020), against 1% of matched, and 7% only open after Jul 2020.
- **Months active**: unmatched storefronts are open 25 of the 36 months (median 24) against 35 (median 36). 41% of unmatched close and 17% open during
  2019-2021, against 5% and 4% of matched. [1e'] has almost no gap months (a license counts as operating in every month between two samples that list it),
  and gaps in sales are rare and not different (0.15 vs. 0.25 months, p = 0.32).
- **Sales**: median monthly sales are $169k for matched storefronts and $32 for unmatched (mean $228k vs. $41k), and 93% vs. 39% of open months have sales.
- **Address ambiguity doesn't explain it**: suite or unit in the address (44% vs. 44%), highway or route addresses (19% vs. 20%), exact Census geocoder
  match (59% vs. 51%, p = 0.18), number of licenses and tradenames at the address, and businesses within 150 m (median 24 vs. 26) are all about the same.
  Matched storefronts have slightly more businesses within 50 m (median 4 vs. 3, rank-sum p = 0.02).
- **Neighborhood**: unmatched storefronts are somewhat more often in rural or low-density block groups (mean urban share 0.69 vs. 0.83, p = 0.002; median
  block-group density 1,229 vs. 2,324 per square mile, rank-sum p = 0.02), though the place type (city, CDP, outside) is the same.
- **All together** (linear probability model, robust SEs, with and without county fixed effects): cohort, open months, and sales carry the differences.
  Address features, surrounding businesses, and place type are all insignificant. Block-group density is borderline (p = 0.06-0.12 once sales or county
  fixed effects are in).
- **Storefront-months**: coverage rises from 53% (2019) to 65% (2020) and 67% (2021). Covered months almost always have sales (98%) and uncovered ones much
  less often (70%), but **within a matched storefront, a month's sales don't predict whether it's covered** (storefront and month fixed effects: 0.008 per
  log point, p = 0.66), and neither do the first or last 3 open months. Coverage gaps inside matched stores look like gaps in Veraset / SafeGraph, not slow
  months.
- **Storefront size**: deferred. `review/9c_storefront_size_checklist.csv` has 10 matched and 10 unmatched storefronts with Google Maps links to check by hand.

**Takeaway**: the unmatched storefronts are mostly stores that barely operated in 2019-2021 by the LCB's own sales records (newer, shorter-lived, little or no
sales), not established stores at ambiguous addresses or in different kinds of places. Gating the store universe by sales removes about 40% of them.

**Open issues**
- Rerun Part A with `store_history_2018_2021.zip` once it's in `raw-data/` (and line it up with Henry's sales gating).
- Sales are self-reported and stop after Aug 2020, so 2020-08 - 2021-12 can't be gated this way.
- Surrounding businesses come from the current SafeGraph POI file, not 2019-2021.
- ACS place estimates for small CDPs have wide margins of error; 58 places have no median income estimate.